# PapyrusLab E03 — run `prep-46527-zp3` (segmento `pherc0814-46527`, offset `zp3`)

Notebook generato da `scripts/build_e03_notebooks.py` (non modificare a mano). Piano congelato: `docs/plans/2026-09-07-e03-tolleranza-offset-z.md`.
Ogni controllo di arresto del piano è un'asserzione: se fallisce, il run si ferma e il log dice dove.
Fanno fede il piano e `configs/e03/offsets.json`; questo generatore ne è solo l'espansione meccanica.

- Output persistiti: `/kaggle/working/e03/out` e `/kaggle/working/e03/logs`
- File pesanti (codice, checkpoint, label, input, cache), non persistiti: `/tmp/e03`


In [ ]:
MODE = "prep-46527-zp3"
KIND = "prep"                  # prep | infer
SEG = "pherc0814-46527"                    # nome della label, es. pherc0814-46527
SHORT = "46527"
SEED = None                    # None nel prep
TAG = "zp3"                    # z13 | zm3 | zp3 (prep) oppure zm5..zp5 (infer)
K = None                          # offset in slice poolate (None nel prep)
Z_START = 25              # primo piano sorgente della finestra di 84
SOURCE_Z_SLICE = [25, 109]
LAYER_ARGS = ""      # "" per la finestra di default, oppure --layer-start S --layer-end E
EXPECTED_INDICES = null
SETS = "held,train"                # i due segmenti di sviluppo hanno entrambi gli insiemi
WORK = "/kaggle/working/e03"
HEAVY = "/tmp/e03"
SRC_URL = "https://vesuvius-challenge-open-data.s3.amazonaws.com/PHerc0814/segments/20260226000000-46527_2um_try2/surface-volumes/2.399um-0.22m-78keV-volume-20260309142202.zarr"
LABEL_SHAPE = [21, 2130, 3455]
TORCH_EXPECTED = "2.10.0+cpu"
LABEL_TREE_SHA256 = "5659236870d7d0408e330f05f6275bd821fc1d7bdea8c9c8f072dfd4ae8b54f0"
LABEL_FILES, LABEL_BYTES = 1386, 118869
LABEL_ALLOWLIST = {"pherc0139-w016": "a62d3e0ecfc9305758fae3bc0d74d99ecf675bcf846aa910ee4a876ee26ccfd5", "pherc0814-46527": "5659236870d7d0408e330f05f6275bd821fc1d7bdea8c9c8f072dfd4ae8b54f0"}      # nome -> impronta congelata: il sigillo non dipende dai nomi
INPUT_TREE_SHA256 = ""       # input da usare in questo run (ufficiale o spostato)
OFFICIAL_INPUT_TREE_SHA256 = "bc7423431221bf24b247a8ba80d264b0306f816c52b4ecc0d08115a82305ac52"
SEALED_SEGMENT = "pherc1667-w029"
assert SEG in LABEL_ALLOWLIST and SEG != SEALED_SEGMENT, f"STOP: {SEG} non e' un segmento di sviluppo"
print("MODE", MODE, "SEG", SEG, "SEED", SEED, "TAG", TAG, "K", K, "z_start", Z_START, "layer", LAYER_ARGS or "default")


In [ ]:
# Guardia della lista bianca (revisione R1, finding 4): prima di qualunque lettura di maschera, la cartella
# delle label montata deve essere uno dei due segmenti di sviluppo E con l'impronta congelata. Il sigillo di
# pherc1667-w029 non dipende dal nome di una cartella.
import glob, hashlib, os

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), len(per_file)

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count("/") - 2
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + "/", "(", len(files), "file )")
# Nessun artefatto del segmento sigillato deve stare dentro il perimetro del notebook: il dataset delle label
# di E03 contiene i soli due segmenti di sviluppo (revisione R2, finding 1).
sealed_hits = glob.glob(f"/kaggle/input/**/*{SEALED_SEGMENT}*", recursive=True)
assert not sealed_hits, f"STOP: il segmento sigillato {SEALED_SEGMENT} risulta montato: {sealed_hits[:3]}"
lab_hits = glob.glob(f"/kaggle/input/**/{SEG}/{SEG}_inklabels.zarr/0/.zarray", recursive=True)
assert lab_hits, f"STOP: label {SEG} non montata sotto /kaggle/input"
LABEL_DIR_MOUNTED = os.path.dirname(os.path.dirname(os.path.dirname(lab_hits[0])))
# Prima i controlli che non leggono byte: nome, collegamenti, voci attese (revisione R2, finding 2).
assert os.path.basename(LABEL_DIR_MOUNTED) in LABEL_ALLOWLIST, f"STOP: {LABEL_DIR_MOUNTED} fuori dalla lista bianca"
assert not os.path.islink(LABEL_DIR_MOUNTED), "STOP: la cartella delle label e' un collegamento"
allowed_top = {f"{SEG}_{k}.zarr" for k in ("inklabels", "supervision_mask", "validation_mask")}
extra = sorted(set(os.listdir(LABEL_DIR_MOUNTED)) - allowed_top)
assert not extra, f"STOP: voci inattese nelle label montate: {extra}"
for d, dirs_, fs in os.walk(LABEL_DIR_MOUNTED):
    for n in dirs_ + fs:
        p = os.path.join(d, n)
        assert not os.path.islink(p), f"STOP: collegamento dentro le label: {p}"
        assert SEALED_SEGMENT not in os.path.relpath(p, LABEL_DIR_MOUNTED), f"STOP: percorso sigillato: {p}"
lsha, lfiles = tree_sha256(LABEL_DIR_MOUNTED)          # solo ora si leggono i byte
print("label montata:", LABEL_DIR_MOUNTED, "file", lfiles, "tree_sha256", lsha)
assert lsha == LABEL_ALLOWLIST[SEG] == LABEL_TREE_SHA256, f"STOP: impronta delle label {lsha} diversa da quella congelata"
print("lista bianca superata")


In [ ]:
%%bash
# Passo 1 — radice misurabile, cache e temporanei dirottati, guardia dei 15 GB
set -e
mkdir -p /kaggle/working/e03/out /kaggle/working/e03/logs /tmp/e03/tmp /tmp/e03/cache/pip /tmp/e03/cache/hf /tmp/e03/checkpoints /tmp/e03/labels /tmp/e03/input
cat > /kaggle/working/e03/env.sh <<'EOF'
export WORK=/kaggle/working/e03
export HEAVY=/tmp/e03
export TMPDIR=$HEAVY/tmp PIP_CACHE_DIR=$HEAVY/cache/pip HF_HOME=$HEAVY/cache/hf
export LIMIT_BYTES=$((15*1024*1024*1024))
disk_check () {
  local used
  used=$(( $(du -sb "$WORK" | cut -f1) + $(du -sb "$HEAVY" | cut -f1) ))
  echo "spazio_byte=$used ($1)" | tee -a "$WORK/logs/disk_check.log"
  if [ "$used" -gt "$LIMIT_BYTES" ]; then echo "STOP: superati 15 GB ($1)" | tee -a "$WORK/logs/disk_check.log"; exit 1; fi
}
EOF
source /kaggle/working/e03/env.sh
echo "MODE=prep-46527-zp3 SEG=pherc0814-46527 SEED=None start=$(date -u +%FT%TZ)" > $WORK/logs/run_info.txt
disk_check "inizio"
df -h /kaggle/working /tmp | tail -2


In [ ]:
# Passo 1 (segue) — versioni dell'ambiente e rete verso le sorgenti
import sys, platform, json, urllib.request, torch
env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0}
print(json.dumps(env, indent=1))
json.dump(env, open(f"{WORK}/logs/env_before_install.json", "w"), indent=1)
assert env["torch"] == TORCH_EXPECTED, f"STOP: PyTorch inatteso {env['torch']} (atteso {TORCH_EXPECTED}): Kaggle ha cambiato immagine, aggiornare il piano"
if KIND == "prep":
    assert not env["cuda_available"], "STOP: il run prep deve girare senza acceleratore"
else:
    assert env["cuda_available"], "STOP: run GPU senza CUDA disponibile"
urls = ["https://huggingface.co/api/models/scrollprize/ink_9um", "https://huggingface.co/api/buckets/scrollprize/datasets"]
if KIND == "prep":
    urls.insert(0, SRC_URL + "/2/.zarray")
for url in urls:
    with urllib.request.urlopen(url, timeout=30) as r:
        print(r.status, url[:90]); assert r.status == 200, f"STOP: rete non raggiunge {url}"


In [ ]:
%%bash
# Passo 2 — checkout parziale di villa al commit congelato
set -e
source /kaggle/working/e03/env.sh
cd $HEAVY
[ -d villa/.git ] || git clone -q --filter=blob:none --no-checkout https://github.com/ScrollPrize/villa.git
cd villa
git sparse-checkout init --cone >/dev/null
git sparse-checkout set ink-detection vesuvius >/dev/null
git checkout -q 3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e
HEAD=$(git rev-parse HEAD); echo "villa HEAD=$HEAD" | tee $WORK/logs/villa_commit.txt
[ "$HEAD" = "3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e" ] || { echo "STOP: commit villa diverso"; exit 1; }
ls -l ink-detection/koine_machines/inference/infer.py ink-detection/scripts/prepare_9um_isotropic_input.py vesuvius/pyproject.toml ink-detection/uv.lock
sha256sum ink-detection/scripts/prepare_9um_isotropic_input.py | tee $WORK/logs/prepare_script_sha256.txt
disk_check "dopo checkout"


In [ ]:
# Passo 3 (prep) — sole dipendenze del pooling ufficiale: zarr 2.18.7 e numcodecs 0.15.1 (come E00), niente altro
import subprocess, sys, json, re
PY = sys.executable
torch_before = subprocess.run([PY, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True).stdout.strip()
r = subprocess.run([PY, "-m", "pip", "install", "--no-deps", "zarr==2.18.7", "numcodecs==0.15.1"], capture_output=True, text=True)
assert r.returncode == 0, "STOP: pip install zarr/numcodecs fallita\n" + r.stderr[-3000:]
# Tentativo 1 di prep-46527 (7 settembre 2026): zarr 2.18.7 importa `asciitree`, assente sull'immagine CPU di Kaggle.
# Stesso ciclo di E00: ogni modulo mancante si installa con --no-deps nella versione del lock di villa (max 10).
lock = open(f"{HEAVY}/villa/ink-detection/uv.lock", encoding="utf-8").read()
def locked_version(dist):
    m = re.search(r'\[\[package\]\]\nname = "' + re.escape(dist.lower()) + r'"\nversion = "([^"]+)"', lock)
    return m.group(1) if m else None
added = []
for attempt in range(10):
    chk = subprocess.run([PY, "-c", "import zarr, numcodecs, numpy, fsspec, aiohttp; print(zarr.__version__, numcodecs.__version__, numpy.__version__, fsspec.__version__, aiohttp.__version__)"], capture_output=True, text=True)
    if chk.returncode == 0:
        break
    m = re.search(r"No module named '([^'.]+)", chk.stderr)
    assert m, "STOP: import fallito per motivo diverso da modulo mancante\n" + chk.stderr[-2000:]
    mod = m.group(1); assert re.fullmatch(r"[A-Za-z0-9_]+", mod), mod
    dist = next((c for c in (mod, mod.replace("_", "-")) if locked_version(c)), None)
    assert dist, f"STOP: modulo mancante '{mod}' non presente in uv.lock"
    r2 = subprocess.run([PY, "-m", "pip", "install", "--no-deps", f"{dist}=={locked_version(dist)}"], capture_output=True, text=True)
    assert r2.returncode == 0, f"STOP: pip install {dist} fallita\n" + r2.stderr[-2000:]
    added.append({"module": mod, "dist": dist, "version": locked_version(dist)}); print("aggiunto", dist, locked_version(dist))
assert chk.returncode == 0, "STOP: import ancora fallito dopo i tentativi ammessi\n" + chk.stderr[-2000:]
torch_after = subprocess.run([PY, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True).stdout.strip()
assert torch_after == torch_before == TORCH_EXPECTED, f"STOP: PyTorch cambiato da {torch_before} a {torch_after}"
vers = chk.stdout.strip().split()
assert vers[0] == "2.18.7", f"STOP: zarr {vers[0]} invece di 2.18.7"
info = {"torch_before": torch_before, "torch_after": torch_after, "zarr": vers[0], "numcodecs": vers[1], "numpy": vers[2], "fsspec": vers[3], "aiohttp": vers[4], "added_packages": added}
json.dump(info, open(f"{WORK}/logs/install.json", "w"), indent=1); print(info)


In [ ]:
# Passo 4 (segue) — label del segmento dal dataset Kaggle privato (ricerca ricorsiva: il mount cambia fra sessioni
# CPU e GPU, lezione di E00), verificata file per file contro manifest.json e per contenuto (tree_sha256);
# ripiego: download diretto dal bucket con 4 thread e attesa crescente (HTTP 429), come E00.
import hashlib, os, shutil, json, glob, re, time, urllib.request, concurrent.futures as cf
PREFIX = f"ink_9um/labels/aligned-scrollprizeorg-21slices/{SEG}/"
API = "https://huggingface.co/api/buckets/scrollprize/datasets/tree/" + PREFIX.rstrip("/")
RESOLVE = "https://huggingface.co/buckets/scrollprize/datasets/resolve/"
DEST = f"{HEAVY}/labels/{SEG}"

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), per_file

def check_label_tree(dest, source_desc):
    n = sum(len(fs) for _, _, fs in os.walk(dest)); b = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(dest) for f in fs)
    assert (n, b) == (LABEL_FILES, LABEL_BYTES), f"STOP: label con {n} file / {b} byte, attesi {(LABEL_FILES, LABEL_BYTES)}"
    tsha, _ = tree_sha256(dest)
    assert tsha == LABEL_TREE_SHA256, f"STOP: contenuto della label diverso da quello congelato (tree sha256 {tsha})"
    open(f"{WORK}/logs/label_count.txt", "w").write(f"seg={SEG} file={n} byte={b} tree_sha256={tsha} source={source_desc}\n")
    za = json.load(open(f"{dest}/{SEG}_inklabels.zarr/0/.zarray")); open(f"{WORK}/logs/label_zarray.json", "w").write(json.dumps(za))
    assert za["shape"] == LABEL_SHAPE, f"STOP: forma della label {za['shape']} diversa da {LABEL_SHAPE}"
    print(f"label verificata: file={n} byte={b} tree_sha256={tsha} ({source_desc}) shape={za['shape']}")

roots = glob.glob(f"/kaggle/input/**/{SEG}/{SEG}_inklabels.zarr/0/.zarray", recursive=True)
manifests = [m for m in glob.glob("/kaggle/input/**/manifest.json", recursive=True) if SEG in json.load(open(m)).get("segments", {})]
print("label montata:", roots[:1], "| manifest:", manifests[:1])
if os.path.isdir(DEST):
    shutil.rmtree(DEST)
if roots and manifests:
    src = os.path.dirname(os.path.dirname(os.path.dirname(roots[0])))       # .../<SEG>
    man = json.load(open(manifests[0]))["segments"][SEG]
    files = man["files"]
    assert (len(files), sum(f["size"] for f in files)) == (LABEL_FILES, LABEL_BYTES), "STOP: manifest del dataset diverso dalle costanti congelate"
    missing = [f["path"] for f in files if not (os.path.isfile(os.path.join(src, f["path"][len(PREFIX):])) and os.path.getsize(os.path.join(src, f["path"][len(PREFIX):])) == f["size"])]
    assert not missing, f"STOP: {len(missing)} file della label mancanti o di dimensione diversa, p.es. {missing[:3]}"
    shutil.copytree(src, DEST)
    check_label_tree(DEST, f"kaggle_dataset manifest_sha256={hashlib.sha256(open(manifests[0], 'rb').read()).hexdigest()}")
else:
    assert KIND == "prep", "STOP: nei run GPU la label deve essere montata (nessun ripiego di rete con la GPU allocata)"
    def list_label_files():
        files, url = [], API
        while url:
            req = urllib.request.Request(url, headers={"User-Agent": "papyruslab-e02"})
            with urllib.request.urlopen(req, timeout=60) as r:
                files += [(e["path"], int(e["size"])) for e in json.load(r) if e.get("type") == "file"]
                m = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", "") or "")
                url = m.group(1) if m else None
        return files
    def fetch(item):
        path, size = item
        assert path.startswith(PREFIX) and ".." not in path, f"STOP: percorso inatteso {path}"
        out = os.path.join(DEST, path[len(PREFIX):])
        if os.path.exists(out) and os.path.getsize(out) == size:
            return size
        os.makedirs(os.path.dirname(out), exist_ok=True)
        last = None
        for attempt in range(8):
            try:
                req = urllib.request.Request(RESOLVE + path, headers={"User-Agent": "papyruslab-e02"})
                with urllib.request.urlopen(req, timeout=60) as r:
                    data = r.read()
                if len(data) == size:
                    open(out, "wb").write(data); return size
                last = f"dimensione {len(data)} != {size}"
            except Exception as ex:
                last = ex
            time.sleep(min(60, 5 * 2 ** attempt))
        raise RuntimeError(f"STOP: download fallito per {path}: {last}")
    files = list_label_files(); total = sum(s for _, s in files)
    assert (len(files), total) == (LABEL_FILES, LABEL_BYTES), f"STOP: label diversa dalla misura congelata: {len(files)} file, {total} byte"
    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=4) as ex:
        got = sum(ex.map(fetch, files))
    print(f"scaricati {got} byte in {time.time() - t0:.0f} s")
    check_label_tree(DEST, "direct_download")


In [ ]:
# Passo 8 (prep) — pooling con finestra sorgente spostata, con verifica del manifest di sorgente (piano 2b)
import base64, hashlib, json, os, subprocess, sys, time
os.makedirs(f"{HEAVY}/repo/scripts", exist_ok=True); os.makedirs(f"{HEAVY}/repo/configs/e03", exist_ok=True)
open(f"{HEAVY}/repo/scripts/e03_pool_shifted.py", "w", encoding="utf-8", newline="\n").write(
    base64.b64decode("IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQb29sIGEgMi40IHVtIHN1cmZhY2Ugdm9sdW1lIHRvIHRoZSB+OS42IHVtIGlzb3Ryb3BpYyAyMS1zbGljZSBpbnB1dCwgd2l0aCBhIHdob2xlLXNsaWNlIFogc2hpZnQuCgpGYWl0aGZ1bCByZXByb2R1Y3Rpb24gb2YgU2Nyb2xsUHJpemUvdmlsbGEgYGluay1kZXRlY3Rpb24vc2NyaXB0cy9wcmVwYXJlXzl1bV9pc290cm9waWNfaW5wdXQucHlgIGF0IGNvbW1pdAozZWExN2Y1NGE5YjNkNWZkMWFhZjczZTFkMmM4Mzg2ZGJhYTlmMzBlIChTSEEtMjU2IDNhZmZmNGYyNDBmZjZlMDkyMjc1MDJmMzE1OGFmMmI5ZDkwODg4ZTllNjQ2YTM4OGQ0MTQ3N2E5MTJkZDAxYWYpLAp3aXRoIG9uZSBhZGRpdGlvbiByZXF1aXJlZCBieSBFMDM6IGAtLXotc3RhcnRgLCB3aGljaCBtb3ZlcyB0aGUgODQtcGxhbmUgc291cmNlIHdpbmRvdyBieSBhIHdob2xlIG51bWJlciBvZgpwb29sZWQgc2xpY2VzICg0IHNvdXJjZSBwbGFuZXMgZWFjaCkuIEF0IGAtLXotc3RhcnQgMTNgICh0aGUgZGVmYXVsdCwgYGNlaWwoKDEwOS04NCkvMilgKSB0aGlzIHNjcmlwdCBtdXN0IGJlCmJ5dGUtaWRlbnRpY2FsIHRvIHRoZSBvZmZpY2lhbCBvbmU6IHNhbWUgdGlsaW5nLCBzYW1lIGZsb2F0MzIgbWVhbiwgc2FtZSBucC5yaW50LCBzYW1lIGNodW5rcywgc2FtZSBjb21wcmVzc29yLApzYW1lIGF0dHJpYnV0ZXMuIFRoYXQgaWRlbnRpdHkgaXMgdmVyaWZpZWQgYWdhaW5zdCB0aGUgZnJvemVuIGZpbmdlcnByaW50IG9mIEUwMgoocGhlcmMwODE0LTQ2NTI3OiBiYzc0MjM0MzEyMjFiZjI0YjI0N2E4YmE4MGQyNjRiMDMwNmY4MTZjNTJiNGVjYzBkMDgxMTVhODIzMDVhYzUyKSBiZWZvcmUgYW55IHNoaWZ0ZWQgaW5wdXQgaXMgdXNlZC4KClZpbGxhJ3MgY29kZSBpcyBmcm96ZW4gZm9yIFBhcHlydXNMYWIgKEUwMC1FMDIgcmVwcm9kdWNpYmlsaXR5KTogaXQgaXMgcmVwcm9kdWNlZCBoZXJlLCBuZXZlciBtb2RpZmllZC4KT3JpZ2luYWwgbGljZW5jZTogc2VlIGh0dHBzOi8vZ2l0aHViLmNvbS9TY3JvbGxQcml6ZS92aWxsYS4KClNvdXJjZSBtYW5pZmVzdCAocGxhbiBzdGVwIDJiLCByZXZpZXcgUjEgZmluZGluZyAyKS4gVGhlIG9mZmljaWFsIHBvb2xpbmcgb25seSByZWFkcyBzb3VyY2UgcGxhbmVzIDEzLTk2LCBzbyB0aGUKcGxhbmVzIDEtMTIgYW5kIDk3LTEwOCAtLSB3aGljaCBmZWVkIHRoZSB0aHJlZSBuZXcgc2xpY2VzIG9mIGV2ZXJ5IHNoaWZ0ZWQgaW5wdXQgLS0gaGF2ZSBubyBmcm96ZW4gaWRlbnRpdHkuIEF0CmxldmVsIDIgYSBjaHVuayBpcyBbMTA5LCAxMjgsIDEyOF06IHRoZSB3aG9sZSBkZXB0aCB0cmF2ZWxzIGFueXdheSwgc28gaGFzaGluZyB0aGUgZnVsbC1kZXB0aCBibG9jayBwZXIgdGlsZSBjb3N0cwpubyBleHRyYSB0cmFmZmljLiBgLS1zb3VyY2UtbWFuaWZlc3RgIHdyaXRlcyB0aG9zZSBwZXItdGlsZSBoYXNoZXMsIGAtLXZlcmlmeS1zb3VyY2UtbWFuaWZlc3RgIGNoZWNrcyB0aGVtIGFuZApzdG9wcyBhdCB0aGUgZmlyc3QgZGlmZmVyZW5jZS4KClVzYWdlOgogIHB5dGhvbiBzY3JpcHRzL2UwM19wb29sX3NoaWZ0ZWQucHkgPGlucHV0X3phcnJ8VVJMPiA8b3V0cHV0LnphcnI+IFstLWxldmVsIDJdIFstLXdvcmtlcnMgNF0gWy0tei1zdGFydCAxM10KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIFstLXNlZ21lbnQgTkFNRV0gWy0tc291cmNlLW1hbmlmZXN0IE9VVC5qc29uXSBbLS12ZXJpZnktc291cmNlLW1hbmlmZXN0IElOLmpzb25dCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHRocmVhZGluZwpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgVGhyZWFkUG9vbEV4ZWN1dG9yCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucApmcm9tIG51bWNvZGVjcyBpbXBvcnQgQmxvc2MKaW1wb3J0IHphcnIKClZFUlNJT04gPSAiZTAzX3Bvb2xfc2hpZnRlZC8xLjAiCk9VVFBVVF9aID0gMjEKUE9PTF9aID0gNApJTlBVVF9aID0gT1VUUFVUX1ogKiBQT09MX1ogICAgICAgICAgIyA4NApUSUxFID0gNTEyCkNIVU5LX1hZID0gMTI4Ck9GRklDSUFMX0ZPUk1BVCA9ICJsZXZlbDItem1lYW40LTIxc2xpY2UtdjEiCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gdmlsbGEsIHJlcHJvZHVjZWQKZGVmIGNlbnRlcmVkX3NsaWNlKGxlbmd0aDogaW50LCByZXF1ZXN0ZWQ6IGludCkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgaWYgcmVxdWVzdGVkID4gbGVuZ3RoOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDYW5ub3QgdGFrZSB7cmVxdWVzdGVkfSBjZW50ZXJlZCBwbGFuZXMgZnJvbSB7bGVuZ3RofSIpCiAgICBzdGFydCA9IG1hdGguY2VpbCgobGVuZ3RoIC0gcmVxdWVzdGVkKSAvIDIpCiAgICByZXR1cm4gc3RhcnQsIHN0YXJ0ICsgcmVxdWVzdGVkCgoKZGVmIG9wZW5fc291cmNlX2FycmF5KHBhdGg6IHN0ciwgbGV2ZWw6IHN0cikgLT4gemFyci5BcnJheToKICAgIG5vZGUgPSB6YXJyLm9wZW4ocGF0aCwgbW9kZT0iciIpCiAgICBpZiBpc2luc3RhbmNlKG5vZGUsIHphcnIuQXJyYXkpOgogICAgICAgIHJldHVybiBub2RlCiAgICBpZiBsZXZlbCBub3QgaW4gbm9kZToKICAgICAgICBhdmFpbGFibGUgPSBzb3J0ZWQobm9kZS5hcnJheV9rZXlzKCkpCiAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJQeXJhbWlkIGxldmVsICd7bGV2ZWx9JyBub3QgZm91bmQgaW4ge3BhdGh9OyBhdmFpbGFibGU6IHthdmFpbGFibGV9IikKICAgIHJldHVybiBub2RlW2xldmVsXQoKCmRlZiBwb29sX2Jsb2NrKGZ1bGw6IG5wLm5kYXJyYXksIHowOiBpbnQsIHoxOiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJUaGUgb2ZmaWNpYWwgYXJpdGhtZXRpYzogZmxvYXQzMiBtZWFuIG9mIDQgcGxhbmVzLCBucC5yaW50LCB1aW50OC4gYGZ1bGxgIGlzIHRoZSBmdWxsLWRlcHRoIGJsb2NrLiIiIgogICAgYmxvY2sgPSBmdWxsW3owOnoxXS5hc3R5cGUobnAuZmxvYXQzMikKICAgIHJldHVybiBucC5yaW50KGJsb2NrLnJlc2hhcGUoT1VUUFVUX1osIFBPT0xfWiwgYmxvY2suc2hhcGVbMV0sIGJsb2NrLnNoYXBlWzJdKS5tZWFuKGF4aXM9MSkpLmFzdHlwZShucC51aW50OCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBzb3VyY2UgbWFuaWZlc3QKZGVmIHRpbGVfa2V5KHkwOiBpbnQsIHgwOiBpbnQpIC0+IHN0cjoKICAgIHJldHVybiBmInt5MH1fe3gwfSIKCgpkZWYgdGlsZV9oYXNoKGZ1bGw6IG5wLm5kYXJyYXkpIC0+IHN0cjoKICAgICIiIlNIQS0yNTYgb2YgdGhlIGZ1bGwtZGVwdGggc291cmNlIGJsb2NrLCBDLWNvbnRpZ3VvdXMgdWludDg6IGNvdmVycyBldmVyeSBwbGFuZSwgMC4uc2hhcGVbMF0tMS4iIiIKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihucC5hc2NvbnRpZ3VvdXNhcnJheShmdWxsLCBkdHlwZT1ucC51aW50OCkudG9ieXRlcygpKS5oZXhkaWdlc3QoKQoKCmRlZiBsb2FkX21hbmlmZXN0KHBhdGg6IFBhdGgsIHNlZ21lbnQ6IHN0ciwgZXhwZWN0OiBkaWN0KSAtPiBkaWN0OgogICAgZG9jID0ganNvbi5sb2FkcyhQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHNlZ21lbnRzID0gZG9jLmdldCgic2VnbWVudHMiLCB7fSkKICAgIGlmIHNlZ21lbnQgbm90IGluIHNlZ21lbnRzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJTVE9QOiBtYW5pZmVzdCB7cGF0aH0gaGFzIG5vIGVudHJ5IGZvciBzZWdtZW50ICd7c2VnbWVudH0nICIKICAgICAgICAgICAgICAgICAgICAgICAgIGYiKHByZXNlbnQ6IHtzb3J0ZWQoc2VnbWVudHMpfSkiKQogICAgZW50cnkgPSBzZWdtZW50c1tzZWdtZW50XQogICAgZm9yIGtleSBpbiAoInNvdXJjZSIsICJsZXZlbCIsICJzb3VyY2Vfc2hhcGVfenl4IiwgInRpbGUiKToKICAgICAgICBpZiBlbnRyeS5nZXQoa2V5KSAhPSBleHBlY3Rba2V5XToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlNUT1A6IG1hbmlmZXN0IHtwYXRofSBkaXNhZ3JlZXMgb24gJ3trZXl9JzogIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYibWFuaWZlc3Qge2VudHJ5LmdldChrZXkpIXJ9IHZzIHNvdXJjZSB7ZXhwZWN0W2tleV0hcn0iKQogICAgcmV0dXJuIGVudHJ5CgoKZGVmIHdyaXRlX21hbmlmZXN0KHBhdGg6IFBhdGgsIHNlZ21lbnQ6IHN0ciwgZW50cnk6IGRpY3QpIC0+IE5vbmU6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgZG9jID0geyJ2ZXJzaW9uIjogVkVSU0lPTiwgInNlZ21lbnRzIjoge319CiAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgIGRvYyA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgZG9jLnNldGRlZmF1bHQoInNlZ21lbnRzIiwge30pCiAgICAgICAgZG9jWyJ2ZXJzaW9uIl0gPSBWRVJTSU9OCiAgICBkb2NbInNlZ21lbnRzIl1bc2VnbWVudF0gPSBlbnRyeQogICAgZG9jWyJzZWdtZW50cyJdID0ge2s6IGRvY1sic2VnbWVudHMiXVtrXSBmb3IgayBpbiBzb3J0ZWQoZG9jWyJzZWdtZW50cyJdKX0KICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKGRvYywgaW5kZW50PTEsIGVuc3VyZV9hc2NpaT1GYWxzZSwgc29ydF9rZXlzPVRydWUpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBydW4KZGVmIHJ1bihpbnB1dF96YXJyOiBzdHIsIG91dHB1dF96YXJyOiBQYXRoIHwgc3RyLCAqLCBsZXZlbDogc3RyID0gIjIiLCB3b3JrZXJzOiBpbnQgPSA0LAogICAgICAgIHpfc3RhcnQ6IGludCB8IE5vbmUgPSBOb25lLCBzZWdtZW50OiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBtYW5pZmVzdF9vdXQ6IFBhdGggfCBzdHIgfCBOb25lID0gTm9uZSwgbWFuaWZlc3RfdmVyaWZ5OiBQYXRoIHwgc3RyIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Q6CiAgICBvdXRwdXRfemFyciA9IFBhdGgob3V0cHV0X3phcnIpCiAgICBzb3VyY2UgPSBvcGVuX3NvdXJjZV9hcnJheShpbnB1dF96YXJyLCBsZXZlbCkKICAgIHNoYXBlID0gdHVwbGUoaW50KHYpIGZvciB2IGluIHNvdXJjZS5zaGFwZSkKICAgIGlmIHNvdXJjZS5kdHlwZSAhPSBucC51aW50OCBvciBsZW4oc2hhcGUpICE9IDM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIDNEIHVpbnQ4IHNvdXJjZSwgZ290IHtzaGFwZX0ge3NvdXJjZS5kdHlwZX0iKQoKICAgIHpfZGVmYXVsdCwgXyA9IGNlbnRlcmVkX3NsaWNlKHNoYXBlWzBdLCBJTlBVVF9aKQogICAgaWYgel9zdGFydCBpcyBOb25lOgogICAgICAgIHowID0gel9kZWZhdWx0CiAgICBlbHNlOgogICAgICAgIHowID0gaW50KHpfc3RhcnQpCiAgICAgICAgaWYgejAgPCAwIG9yIHowICsgSU5QVVRfWiA+IHNoYXBlWzBdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU1RPUDogLS16LXN0YXJ0IHt6MH0gb3V0IG9mIHJhbmdlOiBpdCBtdXN0IGxpZSBpbiBbMCwge3NoYXBlWzBdIC0gSU5QVVRfWn1dICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmImZvciBhIHNvdXJjZSB3aXRoIHtzaGFwZVswXX0gcGxhbmVzIikKICAgICAgICBpZiAoejAgLSB6X2RlZmF1bHQpICUgUE9PTF9aICE9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJTVE9QOiAtLXotc3RhcnQge3owfSBpcyBub3QgYSB3aG9sZS1zbGljZSBzaGlmdCBmcm9tIHRoZSBvZmZpY2lhbCB7el9kZWZhdWx0fSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZiIoYSBwb29sZWQgc2xpY2UgaXMge1BPT0xfWn0gc291cmNlIHBsYW5lcykiKQogICAgejEgPSB6MCArIElOUFVUX1oKICAgIHNoaWZ0ID0gKHowIC0gel9kZWZhdWx0KSAvLyBQT09MX1oKCiAgICBpZiAobWFuaWZlc3Rfb3V0IG9yIG1hbmlmZXN0X3ZlcmlmeSkgYW5kIG5vdCBzZWdtZW50OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNUT1A6IC0tc2VnbWVudCBpcyByZXF1aXJlZCB0b2dldGhlciB3aXRoIC0tc291cmNlLW1hbmlmZXN0IC8gLS12ZXJpZnktc291cmNlLW1hbmlmZXN0IikKCiAgICBleHBlY3QgPSB7InNvdXJjZSI6IHN0cihpbnB1dF96YXJyKSwgImxldmVsIjogc3RyKGxldmVsKSwgInNvdXJjZV9zaGFwZV96eXgiOiBsaXN0KHNoYXBlKSwgInRpbGUiOiBUSUxFfQogICAgdmVyaWZ5X2VudHJ5ID0gbG9hZF9tYW5pZmVzdChQYXRoKG1hbmlmZXN0X3ZlcmlmeSksIHNlZ21lbnQsIGV4cGVjdCkgaWYgbWFuaWZlc3RfdmVyaWZ5IGVsc2UgTm9uZQoKICAgIHBhcnRpYWwgPSBvdXRwdXRfemFyci53aXRoX25hbWUob3V0cHV0X3phcnIubmFtZSArICIucGFydGlhbCIpCiAgICBpZiBvdXRwdXRfemFyci5leGlzdHMoKSBvciBwYXJ0aWFsLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlJlZnVzaW5nIHRvIHJlcGxhY2Uge291dHB1dF96YXJyfSBvciB7cGFydGlhbH0iKQoKICAgIGF0dHJzID0gewogICAgICAgICJmb3JtYXQiOiBPRkZJQ0lBTF9GT1JNQVQgaWYgc2hpZnQgPT0gMCBlbHNlIE9GRklDSUFMX0ZPUk1BVCArICIrenNoaWZ0IiwKICAgICAgICAic291cmNlIjogc3RyKGlucHV0X3phcnIpLAogICAgICAgICJzb3VyY2VfbGV2ZWwiOiBzdHIobGV2ZWwpLAogICAgICAgICJzb3VyY2Vfc2hhcGVfenl4IjogbGlzdChzaGFwZSksCiAgICAgICAgInNvdXJjZV96X3NsaWNlIjogW3owLCB6MV0sCiAgICAgICAgInpfcG9vbCI6ICJyb3VuZGVkIG1lYW4gb2YgNCBjZW50ZXJlZCBzb3VyY2UgcGxhbmVzIiwKICAgIH0KICAgIGlmIHNoaWZ0ICE9IDA6CiAgICAgICAgYXR0cnNbImUwM196X3NoaWZ0X3NsaWNlcyJdID0gaW50KHNoaWZ0KQoKICAgIGdyb3VwID0gemFyci5vcGVuX2dyb3VwKHN0cihwYXJ0aWFsKSwgbW9kZT0idyIpCiAgICBncm91cC5hdHRycy51cGRhdGUoYXR0cnMpCiAgICB0YXJnZXQgPSBncm91cC5jcmVhdGVfZGF0YXNldCgKICAgICAgICAiMCIsCiAgICAgICAgc2hhcGU9KE9VVFBVVF9aLCBzaGFwZVsxXSwgc2hhcGVbMl0pLAogICAgICAgIGNodW5rcz0oT1VUUFVUX1osIG1pbihDSFVOS19YWSwgc2hhcGVbMV0pLCBtaW4oQ0hVTktfWFksIHNoYXBlWzJdKSksCiAgICAgICAgZHR5cGU9bnAudWludDgsCiAgICAgICAgY29tcHJlc3Nvcj1CbG9zYyhjbmFtZT0ienN0ZCIsIGNsZXZlbD01LCBzaHVmZmxlPUJsb3NjLkJJVFNIVUZGTEUpLAogICAgICAgIGZpbGxfdmFsdWU9MCwKICAgICkKICAgIHRpbGVzID0gWwogICAgICAgICh5MCwgbWluKHNoYXBlWzFdLCB5MCArIFRJTEUpLCB4MCwgbWluKHNoYXBlWzJdLCB4MCArIFRJTEUpKQogICAgICAgIGZvciB5MCBpbiByYW5nZSgwLCBzaGFwZVsxXSwgVElMRSkKICAgICAgICBmb3IgeDAgaW4gcmFuZ2UoMCwgc2hhcGVbMl0sIFRJTEUpCiAgICBdCgogICAgaGFzaGVzOiBkaWN0W3N0ciwgc3RyXSA9IHt9CiAgICBsb2NrID0gdGhyZWFkaW5nLkxvY2soKQoKICAgIGRlZiBwcm9jZXNzKHRpbGU6IHR1cGxlW2ludCwgaW50LCBpbnQsIGludF0pIC0+IGludDoKICAgICAgICB5MCwgeTEsIHgwLCB4MSA9IHRpbGUKICAgICAgICBmdWxsID0gbnAuYXNhcnJheShzb3VyY2VbOiwgeTA6eTEsIHgwOngxXSkgICAgICAgICMgcHJvZm9uZGl0YScgcGllbmE6IHN0ZXNzaSBjaHVuaywgbmVzc3VuIHRyYWZmaWNvIGluIHBpdScKICAgICAgICBpZiBtYW5pZmVzdF9vdXQgb3IgdmVyaWZ5X2VudHJ5IGlzIG5vdCBOb25lOgogICAgICAgICAgICBkaWdlc3QgPSB0aWxlX2hhc2goZnVsbCkKICAgICAgICAgICAga2V5ID0gdGlsZV9rZXkoeTAsIHgwKQogICAgICAgICAgICBpZiB2ZXJpZnlfZW50cnkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBleHBlY3RlZCA9IHZlcmlmeV9lbnRyeVsidGlsZXMiXS5nZXQoa2V5KQogICAgICAgICAgICAgICAgaWYgZXhwZWN0ZWQgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU1RPUDogc291cmNlIG1hbmlmZXN0IGhhcyBubyB0aWxlIHtrZXl9IikKICAgICAgICAgICAgICAgIGlmIGV4cGVjdGVkICE9IGRpZ2VzdDoKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU1RPUDogc291cmNlIG1hbmlmZXN0IG1pc21hdGNoIG9uIHRpbGUge2tleX0gIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZiIocGxhbmVzIDAte3NoYXBlWzBdIC0gMX0pOiB0aGUgc291cmNlIGNoYW5nZWQgdXBzdHJlYW0gb3IgaXMgY29ycnVwdCIpCiAgICAgICAgICAgIHdpdGggbG9jazoKICAgICAgICAgICAgICAgIGhhc2hlc1trZXldID0gZGlnZXN0CiAgICAgICAgdGFyZ2V0WzosIHkwOnkxLCB4MDp4MV0gPSBwb29sX2Jsb2NrKGZ1bGwsIHowLCB6MSkKICAgICAgICByZXR1cm4gMQoKICAgIGNvbXBsZXRlZCA9IDAKICAgIHdpdGggVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPXdvcmtlcnMpIGFzIGV4OgogICAgICAgIGZvciBjb3VudCBpbiBleC5tYXAocHJvY2VzcywgdGlsZXMpOgogICAgICAgICAgICBjb21wbGV0ZWQgKz0gY291bnQKICAgICAgICAgICAgaWYgY29tcGxldGVkICUgNTAgPT0gMCBvciBjb21wbGV0ZWQgPT0gbGVuKHRpbGVzKToKICAgICAgICAgICAgICAgIHByaW50KGYidGlsZXM9e2NvbXBsZXRlZH0ve2xlbih0aWxlcyl9IiwgZmx1c2g9VHJ1ZSkKCiAgICBpZiB2ZXJpZnlfZW50cnkgaXMgbm90IE5vbmU6CiAgICAgICAgbWlzc2luZyA9IHNldCh2ZXJpZnlfZW50cnlbInRpbGVzIl0pIC0gc2V0KGhhc2hlcykKICAgICAgICBpZiBtaXNzaW5nOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU1RPUDogc291cmNlIG1hbmlmZXN0IGxpc3RzIHtsZW4obWlzc2luZyl9IHRpbGVzIG5ldmVyIHJlYWQ6IHtzb3J0ZWQobWlzc2luZylbOjVdfSIpCgogICAgcGFydGlhbC5yZXBsYWNlKG91dHB1dF96YXJyKQogICAgaWYgbWFuaWZlc3Rfb3V0OgogICAgICAgIHdyaXRlX21hbmlmZXN0KFBhdGgobWFuaWZlc3Rfb3V0KSwgc2VnbWVudCwgewogICAgICAgICAgICAic291cmNlIjogc3RyKGlucHV0X3phcnIpLCAibGV2ZWwiOiBzdHIobGV2ZWwpLCAic291cmNlX3NoYXBlX3p5eCI6IGxpc3Qoc2hhcGUpLAogICAgICAgICAgICAidGlsZSI6IFRJTEUsICJoYXNoX29mIjogInNoYTI1NiBvZiB0aGUgQy1jb250aWd1b3VzIHVpbnQ4IGJ5dGVzIG9mIHNvdXJjZVs6LCB5MDp5MSwgeDA6eDFdIChhbGwgcGxhbmVzKSIsCiAgICAgICAgICAgICJjb3ZlcnNfYWxsX3NvdXJjZV9wbGFuZXMiOiBUcnVlLCAibl90aWxlcyI6IGxlbih0aWxlcyksCiAgICAgICAgICAgICJ0aWxlcyI6IHtrOiBoYXNoZXNba10gZm9yIGsgaW4gc29ydGVkKGhhc2hlcyl9LAogICAgICAgICAgICAiZ2VuZXJhdGVkX2F0IjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksICJ2ZXJzaW9uIjogVkVSU0lPTiwKICAgICAgICB9KQogICAgcHJpbnQoZiJ3cm90ZSB7b3V0cHV0X3phcnJ9IHNoYXBlPXt0dXBsZSh0YXJnZXQuc2hhcGUpfSB6X3N0YXJ0PXt6MH0gel9zaGlmdF9zbGljZXM9e3NoaWZ0OitkfSIpCiAgICByZXR1cm4geyJzaGFwZSI6IHR1cGxlKHRhcmdldC5zaGFwZSksICJ6X3N0YXJ0IjogejAsICJ6X3NsaWNlIjogW3owLCB6MV0sICJzaGlmdCI6IHNoaWZ0LAogICAgICAgICAgICAidGlsZXMiOiBsZW4odGlsZXMpLCAiYXR0cnMiOiBhdHRyc30KCgpkZWYgbWFpbihhcmd2PU5vbmUpIC0+IGludDoKICAgIHAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLCBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgcC5hZGRfYXJndW1lbnQoImlucHV0X3phcnIiLCBoZWxwPSJTdXJmYWNlLXZvbHVtZSBPTUUtWmFyciBncm91cCBwYXRoIG9yIFVSTCAob3IgYSBiYXJlIDNEIGFycmF5KS4iKQogICAgcC5hZGRfYXJndW1lbnQoIm91dHB1dF96YXJyIiwgdHlwZT1QYXRoLCBoZWxwPSJPdXRwdXQgWmFyciBwYXRoOyByZWZ1c2VzIHRvIG92ZXJ3cml0ZS4iKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbGV2ZWwiLCBkZWZhdWx0PSIyIiwgaGVscD0iSW5wdXQgcHlyYW1pZCBsZXZlbCB0byByZWFkIChkZWZhdWx0IDIpLiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS13b3JrZXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXotc3RhcnQiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgaGVscD0iRmlyc3Qgc291cmNlIHBsYW5lIG9mIHRoZSA4NC1wbGFuZSB3aW5kb3c7IGRlZmF1bHQgPSB0aGUgb2ZmaWNpYWwgY2VudHJlZCBzdGFydCAoMTMgb2YgMTA5KS4gIgogICAgICAgICAgICAgICAgICAgICAgICAiTXVzdCBkaWZmZXIgZnJvbSBpdCBieSBhIHdob2xlIG51bWJlciBvZiBwb29sZWQgc2xpY2VzICg0IHBsYW5lcykuIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXNlZ21lbnQiLCBkZWZhdWx0PU5vbmUsIGhlbHA9IlNlZ21lbnQgbmFtZSwgcmVxdWlyZWQgd2l0aCB0aGUgbWFuaWZlc3Qgb3B0aW9ucy4iKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc291cmNlLW1hbmlmZXN0IiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUsIGhlbHA9IldyaXRlIHBlci10aWxlIGZ1bGwtZGVwdGggc291cmNlIGhhc2hlcyBoZXJlLiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS12ZXJpZnktc291cmNlLW1hbmlmZXN0IiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUsIGhlbHA9IlZlcmlmeSBhZ2FpbnN0IHRoaXMgbWFuaWZlc3Qgd2hpbGUgcmVhZGluZy4iKQogICAgYSA9IHAucGFyc2VfYXJncyhhcmd2KQogICAgdHJ5OgogICAgICAgIHJ1bihhLmlucHV0X3phcnIsIGEub3V0cHV0X3phcnIsIGxldmVsPWEubGV2ZWwsIHdvcmtlcnM9YS53b3JrZXJzLCB6X3N0YXJ0PWEuel9zdGFydCwKICAgICAgICAgICAgc2VnbWVudD1hLnNlZ21lbnQsIG1hbmlmZXN0X291dD1hLnNvdXJjZV9tYW5pZmVzdCwgbWFuaWZlc3RfdmVyaWZ5PWEudmVyaWZ5X3NvdXJjZV9tYW5pZmVzdCkKICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgRmlsZUV4aXN0c0Vycm9yLCBLZXlFcnJvcikgYXMgZXhjOgogICAgICAgIHByaW50KHN0cihleGMpLCBmaWxlPV9faW1wb3J0X18oInN5cyIpLnN0ZGVycikKICAgICAgICByZXR1cm4gMwogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==").decode("utf-8"))
MANIFEST_B64 = "ewogInNlZ21lbnRzIjogewogICJwaGVyYzA4MTQtNDY1MjciOiB7CiAgICJjb3ZlcnNfYWxsX3NvdXJjZV9wbGFuZXMiOiB0cnVlLAogICAiZ2VuZXJhdGVkX2F0IjogIjIwMjYtMDktMDdUMTM6MzQ6MjgrMDA6MDAiLAogICAiaGFzaF9vZiI6ICJzaGEyNTYgb2YgdGhlIEMtY29udGlndW91cyB1aW50OCBieXRlcyBvZiBzb3VyY2VbOiwgeTA6eTEsIHgwOngxXSAoYWxsIHBsYW5lcykiLAogICAibGV2ZWwiOiAiMiIsCiAgICJuX3RpbGVzIjogMzUsCiAgICJzb3VyY2UiOiAiaHR0cHM6Ly92ZXN1dml1cy1jaGFsbGVuZ2Utb3Blbi1kYXRhLnMzLmFtYXpvbmF3cy5jb20vUEhlcmMwODE0L3NlZ21lbnRzLzIwMjYwMjI2MDAwMDAwLTQ2NTI3XzJ1bV90cnkyL3N1cmZhY2Utdm9sdW1lcy8yLjM5OXVtLTAuMjJtLTc4a2VWLXZvbHVtZS0yMDI2MDMwOTE0MjIwMi56YXJyIiwKICAgInNvdXJjZV9zaGFwZV96eXgiOiBbCiAgICAxMDksCiAgICAyMTMwLAogICAgMzQ1NQogICBdLAogICAidGlsZSI6IDUxMiwKICAgInRpbGVzIjogewogICAgIjBfMCI6ICI2YWQxYzA2MjIxYjc0MGExMjBkNGU2ZTEzZWZiODQ0NzU2NmE5OTU5MWI4MGQzNmVlYTY3ZjNmNGFiZWFlMzVlIiwKICAgICIwXzEwMjQiOiAiMDhkZjgzZWZjMTNiZTBiYThjYjFmZGM3ZjcyOGJmMjE5NzFlNTU0MzU4MjhiNjllZjc4NTMyYjRkM2VmYmI4ZCIsCiAgICAiMF8xNTM2IjogIjNiNGFmOThlMWYxZDIxYTE2YjBkM2I3MzNhNmQ1MDEyYzM2NjE3YTVjN2RkZWNmYTkwYWE1OTZiMzcwZGE0MmYiLAogICAgIjBfMjA0OCI6ICIxZDQ5ZGRlMWExN2IyMmVjNjA0ODA4Mjk2NGNiMmQwYTU3MTE4YWUxOGI5ZmY2NTI3NTM0ZWRkMjU5MjU1NzEzIiwKICAgICIwXzI1NjAiOiAiNDZkMzViOGM0NWFiZjMwZGZlYWJiMGQ3MjY5MDQwZDM5N2U3ZTYyZTAzOWE5YmE2YWQ5NmI3ZDQ1OGJhNzdhMyIsCiAgICAiMF8zMDcyIjogImE4MjY5NzAwY2E1N2QxZjUyYWM3NjJjYWQ4N2ZjMTI0ZDM4ZDdlMTcwOTJjNmNiNDAwNGI2OTY3OTEyNzUzMjAiLAogICAgIjBfNTEyIjogIjdjYzY3YTc2NDBhNWYyNzQ5MzgxOTA4MGE5OGRiOGRhMzNjODRhMGExOGZjMzdhNTVjMDM2MTBlN2Y2MjRhZTAiLAogICAgIjEwMjRfMCI6ICJjNjg1MDc0YWFkODFiY2IxMmM0Zjc3MjMzOGI3NmU1MzFmZjNiMzM0MTBiN2ZlZmQwZjllYTRhNzI5YjdhM2QxIiwKICAgICIxMDI0XzEwMjQiOiAiZTgxM2MwOWM5ODZmYzRlYjRhNGQ2NGRmZWMxNzhjMzJkNDBlNzI0NTE2OTcwNDA1MTdiMGI3MTFjZjJlOWQwOSIsCiAgICAiMTAyNF8xNTM2IjogImNjM2U3YmZlODVjY2FkMmQ2MDBlZDA5YWE0OTA3MTBhNzNhY2Y3YjNhNWQxYjMzZWRmN2M5YjFkMzI1MGM4ODYiLAogICAgIjEwMjRfMjA0OCI6ICI5ZDEwZWUwNWQ5ODkzNTE4NjMzZWQ4MmM5MDU3YzRhODU3ZDg3OGE1ZjhkNGNkNzY1YzBkMjczZjUxZTU0OThjIiwKICAgICIxMDI0XzI1NjAiOiAiYjI1M2NlZGM2ZmY3YWFjZGRiMzVlMTZjNjIwZWI1NDdiY2Y2MWM0NzZmMjM5NWNhYmE3Y2MwYWIwNmE1ODE4ZCIsCiAgICAiMTAyNF8zMDcyIjogImE0NzFiMTIwZmFiNDE1ZjNmZDZmZDg1NmFmYTMyYWEyZDAzNjY0YTJjZDc2ZjA1MjVhZjc4NDkwYjU2M2U3NmYiLAogICAgIjEwMjRfNTEyIjogIjc0NDdmMzI4NTMzZjg1OTJlMWNlYjM5ZGI5MWJjZGJkZjc3YTFmZTViZmQ5OWEyOWI4YTRiMzg5MjVmOGE0MWYiLAogICAgIjE1MzZfMCI6ICIzMjE3NjRkNDVkODAxMzg0NGNjYmRkYzllMDE2MjQ0ZGFkYWFiYzRmNjA5YzkxZTJiYTEyODY4NzJiNzRhODg4IiwKICAgICIxNTM2XzEwMjQiOiAiOTdmNzhiZTE3ZjUyODQ0MzQ5OTRiOWRlMWE4ZWE2ZjNjMmQxNmNmMGFlYjQ2Y2I1ZGI4ZjgxMmYyOTJiNjA0ZCIsCiAgICAiMTUzNl8xNTM2IjogIjNmODI2NmY4MjU2YjY5MWEyZGJkMzYwOWUxMzI5NmRjMzYyMzJmMTg1ODFmYmJjZWZkOTEyYmJmOGJhZDU3ODYiLAogICAgIjE1MzZfMjA0OCI6ICIzZjhhY2ZlZGU3ODg3MDRjNWRhZGFjZTQ1NmYyYjBmODY4MDNlOTlmMzdhZTg1Yjg1MGZkZmFmOGM5ZGQzMTE4IiwKICAgICIxNTM2XzI1NjAiOiAiNmFkMWMwNjIyMWI3NDBhMTIwZDRlNmUxM2VmYjg0NDc1NjZhOTk1OTFiODBkMzZlZWE2N2YzZjRhYmVhZTM1ZSIsCiAgICAiMTUzNl8zMDcyIjogImE0NzFiMTIwZmFiNDE1ZjNmZDZmZDg1NmFmYTMyYWEyZDAzNjY0YTJjZDc2ZjA1MjVhZjc4NDkwYjU2M2U3NmYiLAogICAgIjE1MzZfNTEyIjogImE5ZjUzNTZmMDZjZjA1NGMyMzI3M2E5YmRjM2E3OTUyOGE1ZTU4YTYwOTVmMzk3MzdiNGZhMDU5MmQ4MmIwYzciLAogICAgIjIwNDhfMCI6ICI3NzBkMGIxZTI5ZGVkZTQxNDc1YWI1MjU1NzdmYjIwZjkyNWM2YzdiMmMwZGI0ZmVjMWM2ODY4M2Y0ZDgyNzZlIiwKICAgICIyMDQ4XzEwMjQiOiAiZGU5NDk4MjE5NTM5MjdiMWNhYmVhNmVlZmY4YTA2ZmE4YjJlZmJkZjIxY2ZlNmI2ZTVmYjQ5MzRjNGVhMGYxNyIsCiAgICAiMjA0OF8xNTM2IjogImRlOTQ5ODIxOTUzOTI3YjFjYWJlYTZlZWZmOGEwNmZhOGIyZWZiZGYyMWNmZTZiNmU1ZmI0OTM0YzRlYTBmMTciLAogICAgIjIwNDhfMjA0OCI6ICJkZTk0OTgyMTk1MzkyN2IxY2FiZWE2ZWVmZjhhMDZmYThiMmVmYmRmMjFjZmU2YjZlNWZiNDkzNGM0ZWEwZjE3IiwKICAgICIyMDQ4XzI1NjAiOiAiZGU5NDk4MjE5NTM5MjdiMWNhYmVhNmVlZmY4YTA2ZmE4YjJlZmJkZjIxY2ZlNmI2ZTVmYjQ5MzRjNGVhMGYxNyIsCiAgICAiMjA0OF8zMDcyIjogImJlNTg3MmUxNmEyN2JhNzk5YTI0ZGMyMDg2NDQ0YzkwOTFmNDBkZGQ2Njg3ZjkwOGE3OWI4NjBhZGU4ZGU3OWMiLAogICAgIjIwNDhfNTEyIjogImRlOTQ5ODIxOTUzOTI3YjFjYWJlYTZlZWZmOGEwNmZhOGIyZWZiZGYyMWNmZTZiNmU1ZmI0OTM0YzRlYTBmMTciLAogICAgIjUxMl8wIjogImZjYjAzNzMxMDY5ZGJhZmYwZTg3Y2RjZWJkNDU0YWE4ZTljYzdlMmUzOTk4M2UzZTlmODdlMTM0NTM0ZDYxMGEiLAogICAgIjUxMl8xMDI0IjogImE2OGVkOWVhY2JjMjZlNGZlMzEwYzA5ZGJkNTkzMjEwYTczOWFhZWY1MjA4YTFmYTNkNGNiNTlhNTAzYmNjODciLAogICAgIjUxMl8xNTM2IjogIjIxZjU5YTU0ZmJhNmJkMzNkZGUyMjkyZjQzMWUzMTE4NWRhNjhhNGQwYTAyZjZkMmNlMGNjYThlMDhmZjQzYmUiLAogICAgIjUxMl8yMDQ4IjogIjU3NzFjNjZiZTc2YWNjNjRiNGJjYzY1MmFhOTI3ZGY5ZDJjNWY4ZDAwMzEwMWUwYjlhNDQ0NjdmMjRlZWI3MDEiLAogICAgIjUxMl8yNTYwIjogIjQ1OTg0NDQyZDI1MDczNDAxZWQxYzFhYjJlNDViMmVkNWEyN2FlNzRhYWFjM2YxYTJlYmM4MjlkZjBlZTFiYzMiLAogICAgIjUxMl8zMDcyIjogImYxY2Q0ZWQ2M2VlNmE0YjEyNDI5YTY1NTAxMGRjMzdjMWVlNzJkMDMxNzQyODkyYzdmMDdjOWE4MjRkZjBkNGEiLAogICAgIjUxMl81MTIiOiAiMTFmMzJkZjAxYmJhNDI4YzY0Y2M5Zjk4OTFhZWRjZTY3YjAyMjg4OTI4OWFlZjBhMWEzZGUwZGRkYWI2MDIzZCIKICAgfSwKICAgInZlcnNpb24iOiAiZTAzX3Bvb2xfc2hpZnRlZC8xLjAiCiAgfQogfSwKICJ2ZXJzaW9uIjogImUwM19wb29sX3NoaWZ0ZWQvMS4wIgp9"
man_path = f"{HEAVY}/repo/configs/e03/source_manifest.json"
if MANIFEST_B64:
    open(man_path, "w", encoding="utf-8", newline="\n").write(base64.b64decode(MANIFEST_B64).decode("utf-8"))
    man_args = ["--verify-source-manifest", man_path]
    print("manifest di sorgente: verifica")
else:
    man_args = ["--source-manifest", f"{WORK}/out/source_manifest_{SEG}.json"]
    print("manifest di sorgente: produzione (primo pooling di questo segmento)")
out_zarr = f"{HEAVY}/input/{SEG}_pooled.zarr" if TAG == "z13" else f"{HEAVY}/input/{SEG}_pooled_{TAG}.zarr"
os.makedirs(f"{HEAVY}/input", exist_ok=True)
cmd = [sys.executable, f"{HEAVY}/repo/scripts/e03_pool_shifted.py", SRC_URL, out_zarr,
       "--level", "2", "--workers", "4", "--z-start", str(Z_START), "--segment", SEG] + man_args
print(" ".join(cmd), flush=True)
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True, timeout=12600)
dur = int(time.time() - t0)
open(f"{WORK}/logs/prep_{SEG}_{TAG}.log", "w", encoding="utf-8").write(r.stdout + "\n--- stderr ---\n" + r.stderr)
print(r.stdout[-2000:]); print(r.stderr[-2000:] if r.returncode else "")
assert r.returncode == 0, f"STOP: pooling terminato con exit_code={r.returncode} dopo {dur}s"
print(f"durata_s={dur}")


In [ ]:
# Passo 8 (prep, segue) — forma, attributi, uguaglianza slice a slice con l'input ufficiale, tar + impronte
import glob, hashlib, io, json, os, tarfile
import numpy as np, zarr
g = zarr.open(out_zarr, mode="r"); a = g["0"]
lab = zarr.open(f"{LABEL_DIR_MOUNTED}/{SEG}_inklabels.zarr", mode="r")["0"]
print("input", a.shape, a.dtype, "| label", lab.shape, "| attrs", dict(g.attrs))
assert tuple(a.shape) == tuple(lab.shape) == tuple(LABEL_SHAPE), f"STOP: forma {a.shape} vs label {lab.shape}"
assert list(g.attrs["source_z_slice"]) == SOURCE_Z_SLICE and str(g.attrs["source_level"]) == "2"
assert g.attrs["source_shape_zyx"][0] == 109, "STOP: volume sorgente con profondita' diversa da 109"
if TAG == "z13":
    assert g.attrs["format"] == "level2-zmean4-21slice-v1", g.attrs["format"]
else:
    assert g.attrs["e03_z_shift_slices"] == (Z_START - 13) // 4 and g.attrs["format"].endswith("+zshift")
    # uguaglianza slice a slice con l'input UFFICIALE montato dal dataset E02: prova diretta dell'allineamento XY
    hits = glob.glob(f"/kaggle/input/**/{SEG}_pooled.zarr/0/.zarray", recursive=True)
    assert hits, "STOP: input ufficiale non montato: impossibile provare l'uguaglianza slice a slice"
    off_dir = os.path.dirname(os.path.dirname(hits[0]))
    osha, _ = tree_sha256(off_dir)
    assert osha == OFFICIAL_INPUT_TREE_SHA256, f"STOP: input ufficiale montato con impronta {osha}"
    a0 = zarr.open(off_dir, mode="r")["0"]
    lo_s, lo_0 = (3, 0) if Z_START < 13 else (0, 3)
    same = True
    for y in range(0, a.shape[1], 512):
        y1 = min(a.shape[1], y + 512)
        if not np.array_equal(np.asarray(a[lo_s:lo_s + 18, y:y1, :]), np.asarray(a0[lo_0:lo_0 + 18, y:y1, :])):
            same = False; print("DIFFERENZA nel blocco y", y, y1); break
    assert same, "STOP: le 18 slice condivise non coincidono con l'input ufficiale"
    print("uguaglianza slice a slice con l'ufficiale: verificata")

def tree_sha256_dir(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), len(per_file)

tsha, nfiles = tree_sha256_dir(out_zarr)
tar_path = f"{WORK}/out/{os.path.basename(out_zarr)[:-5]}.tar"
with tarfile.open(tar_path, "w") as tar:
    for p in sorted(os.path.relpath(os.path.join(d, f), os.path.dirname(out_zarr))
                    for d, _, fs in os.walk(out_zarr) for f in fs):
        info = tar.gettarinfo(os.path.join(os.path.dirname(out_zarr), p), arcname=p)
        info.mtime = 0; info.uid = info.gid = 0; info.uname = info.gname = ""
        with open(os.path.join(os.path.dirname(out_zarr), p), "rb") as fh:
            tar.addfile(info, fh)
tar_sha = hashlib.sha256(open(tar_path, "rb").read()).hexdigest()
info = {"segment": SEG, "tag": TAG, "z_start": Z_START, "source_z_slice": SOURCE_Z_SLICE,
        "shape": list(a.shape), "tree_sha256": tsha, "files": nfiles,
        "tar_sha256": tar_sha, "tar_bytes": os.path.getsize(tar_path), "attrs": dict(g.attrs)}
json.dump(info, open(f"{WORK}/out/input_{SEG}_{TAG}.json", "w"), indent=1, sort_keys=True)
print(json.dumps(info, indent=1))


In [ ]:
%%bash
# Persistenza — hash di tutto cio' che viene conservato, stato finale
set -e
source /kaggle/working/e03/env.sh
cp /kaggle/working/e03/env.sh $WORK/logs/env.sh.txt
[ -f /kaggle/working/e03_guard.json ] && cp /kaggle/working/e03_guard.json $WORK/logs/guard.json
echo "end=$(date -u +%FT%TZ)" >> $WORK/logs/run_info.txt
disk_check "finale"
cd $WORK && find out logs -type f ! -name SHA256SUMS -print0 | sort -z | xargs -0 sha256sum > out/SHA256SUMS
cat out/SHA256SUMS
echo "persistito: $(du -sh $WORK | cut -f1)"
